In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

print(torch.__version__)

2.10.0+cu128


In [2]:
transform =  transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = True,
    download = True,
    transform = transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root = "./data",
    train = False,
    download = True,
    transform = transform
)

image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label: ", label)

100%|██████████| 170M/170M [31:43<00:00, 89.6kB/s]


Image shape: torch.Size([3, 32, 32])
Label:  6


In [3]:
'''patch_size = 4

patches = image.unfold(1, patch_size, patch_size)
patches = patches.unfold(2, patch_size, patch_size)

print(patches.shape)'''


'patch_size = 4\n\npatches = image.unfold(1, patch_size, patch_size)\npatches = patches.unfold(2, patch_size, patch_size)\n\nprint(patches.shape)'

In [4]:
'''patches = patches.permute(1, 2, 0, 3, 4)

patches = patches.reshape(64, 48)

print(patches.shape)'''


'patches = patches.permute(1, 2, 0, 3, 4)\n\npatches = patches.reshape(64, 48)\n\nprint(patches.shape)'

In [5]:
'''#patch_embedding
d_model = 128

patch_embedding = nn.Linear(48,d_model)
patches = patch_embedding(patches)

print(patches.shape)'''


'#patch_embedding\nd_model = 128\n\npatch_embedding = nn.Linear(48,d_model)\npatches = patch_embedding(patches)\n\nprint(patches.shape)'

In [6]:
#positional embedding
'''
cls_token = nn.Parameter(
    torch.randn(1,128)
)

patches = torch.cat(
    [cls_token,patches], dim = 0  # BERT와 같이 [CLS]토큰 포함 -> [65,128]
)

pos_embedding = nn.Parameter(
    torch.randn(65, 128)
)

patches = patches + pos_embedding


print(patches.shape)'''



'\ncls_token = nn.Parameter(\n    torch.randn(1,128)\n)\n\npatches = torch.cat(\n    [cls_token,patches], dim = 0  # BERT와 같이 [CLS]토큰 포함 -> [65,128]\n)\n\npos_embedding = nn.Parameter(\n    torch.randn(65, 128)\n)\n\npatches = patches + pos_embedding\n\n\nprint(patches.shape)'

In [7]:
#Batch
from torch.utils.data import DataLoader

batch_size=32

train_loader = DataLoader(
    train_dataset,
    batch_size,
    shuffle=True
)

images, labels = next(iter(train_loader))

patches = images.unfold(2,4,4)
patches = patches.unfold(3,4,4)

patches = patches.permute(0,2,3,1,4,5)
patches = patches.reshape(32,64,48)

#patch embedding
d_model = 128
patch_embedding = nn.Linear(48,d_model)
patches = patch_embedding(patches)

#CLS token 추가
cls_token = nn.Parameter(
    torch.randn(1, 1, d_model)
)

cls_tokens = cls_token.expand(batch_size,-1,-1)

patches = torch.cat(
    [cls_tokens, patches], dim = 1
)

#positional_embedding
pos_embedding = nn.Parameter(
    torch.randn(1, 65, d_model)
)

patches = patches + pos_embedding

print(patches.shape)

torch.Size([32, 65, 128])


In [8]:
import math
import torch
import torch.nn.functional as F
import torch.nn as nn

def scaled_dot_product_attention(Q,K,V, mask = None):
    #Q,K,V: (batch, heads, seq_len, d_k) 크기의 행렬
    scores = torch.matmul(Q,K.transpose(-2,-1)) / math.sqrt(Q.size(-1))

    if mask is not None:
        #Decoder를 위한 mask 수정
        scores = scores.masked_fill(mask, float("-inf"))

    attn = F.softmax(scores,dim=-1)
    output = torch.matmul(attn,V)

    return output, attn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0 #head마다 같은 차원을 갖도록 나누어 떨어지는지 확인

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, L_q, _ = q.shape
        _, L_k, _ = k.shape
        Q = self.W_q(q).view(B, L_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)

                # Mask 차원 맞추기
        if mask is not None:

            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)

            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)

            # bool 타입으로 변환
            mask = mask.bool()

        out, attn = scaled_dot_product_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(B, L_q, self.d_model)
        out = self.W_o(out)

        return out, attn

#AddNorm
class AddNorm(nn.Module):
    def __init__(self,d_model,dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)


    def forward(self,x,sublayer_out):
        return self.norm(x+self.dropout(sublayer_out))


#FeedForward
class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model,d_ff) #512 -> 2048
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(d_ff,d_model) #2048 -> 512

    def forward(self,x):
        return self.fc2(self.act(self.fc1(x)))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.addnorm1 = AddNorm(d_model, dropout)
        self.addnorm2 = AddNorm(d_model, dropout)

    def forward(self, x, mask = None):
        out, attn = self.mha(x,x,x, mask)
        x = self.addnorm1(x, out)

        ffn_out = self.ffn(x)
        x = self.addnorm2(x,ffn_out)

        return x, attn


class Encoder(nn.Module):

    def __init__(self,d_model, num_heads,d_ff,num_layers, dropout = 0.1):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):

        attn = None

        for layer in self.layers:
            x, attn = layer(x, mask)

        return x, attn

In [9]:
'''
encoder = Encoder(
    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=4
)

x, attn = encoder(patches)

print(x.shape)

cls_output = x[:, 0]
cls_classification = nn.Linear(128, 10)
logits = cls_classification(cls_output)
print(cls_output)
'''


'\nencoder = Encoder(\n    d_model=128,\n    num_heads=4,\n    d_ff=512,\n    num_layers=4\n)\n\nx, attn = encoder(patches)\n\nprint(x.shape)\n\ncls_output = x[:, 0]\ncls_classification = nn.Linear(128, 10)\nlogits = cls_classification(cls_output)\nprint(cls_output)\n'

In [10]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=4,
        d_model=128,
        num_heads=4,
        d_ff=512,
        num_layers=4,
        num_classes=10
    ):
        super().__init__()

        self.patch_size = patch_size

        self.num_patches = (image_size // patch_size) ** 2
        self.patch_dim = 3 * patch_size * patch_size

        self.patch_embedding = nn.Linear(self.patch_dim,d_model)
        self.cls_token = nn.Parameter(
            torch.randn(1,1,d_model)
        )
        self.pos_embedding = nn.Parameter(
            torch.randn(1,self.num_patches + 1, d_model)
        )

        self.encoder = Encoder(
                d_model,
                num_heads,
                d_ff,
                num_layers
        )

        self.classifier = nn.Linear(d_model, num_classes)


    def forward(self, x):
        batch_size = x.shape[0]

        patches = x.unfold(2,self.patch_size,self.patch_size)
        patches = patches.unfold(3,self.patch_size,self.patch_size)

        patches = patches.permute(0,2,3,1,4,5)
        patches = patches.reshape(batch_size,self.num_patches,self.patch_dim)

        patches = self.patch_embedding(patches)

        cls_token = self.cls_token.expand(batch_size,-1,-1)
        patches = torch.cat(
            [cls_token ,patches], dim = 1
        )

        patches += self.pos_embedding

        x, attn = self.encoder(patches)

        cls_output = x[:,0]
        logits = self.classifier(cls_output)

        return logits, attn 

In [11]:
#test
'''model = VisionTransformer()
logits,attn = model(images)

print(logits.shape)'''

'model = VisionTransformer()\nlogits,attn = model(images)\n\nprint(logits.shape)'

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionTransformer().to(device)

import torch.optim as optim

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=1e-3)
num_epochs = 10

model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
   
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)
        
        logits, attn = model(images)
    
        loss = loss_fn(
            logits, labels
        )
    
        optimizer.zero_grad()
    
        loss.backward()
    
        optimizer.step()

        running_loss += loss.item()
    epoch_loss = running_loss/len(train_loader)
        
    print(
        f"Epoch {epoch}, Loss: {loss.item():.4f}, Loss_everage : {epoch_loss:.4f}"
        )

Epoch 0, Loss: 1.6592, Loss_everage : 1.9341
Epoch 1, Loss: 1.7187, Loss_everage : 1.7070
Epoch 2, Loss: 1.3349, Loss_everage : 1.6313
Epoch 3, Loss: 1.6195, Loss_everage : 1.5826
Epoch 4, Loss: 1.6037, Loss_everage : 1.5390
Epoch 5, Loss: 1.9013, Loss_everage : 1.5035
Epoch 6, Loss: 1.1584, Loss_everage : 1.4575
Epoch 7, Loss: 1.6254, Loss_everage : 1.4370
Epoch 8, Loss: 1.3964, Loss_everage : 1.4032
Epoch 9, Loss: 1.7335, Loss_everage : 1.3994
